In [ ]:
from itertools import islice
from collections import Counter
import numpy as np
import h5py
import pickle

# Open ARCHS4 v2.1 Data
infile = "/home/ajoshi/archs4/data/archs4_gene_human_v2.1.h5"

# - Use read_direct method to improve read performance
with h5py.File(infile, 'r') as hdf5_file:
    data = hdf5_file['data']
    expression = data['expression'][:]

############################################################################################################
# Open and read the gene-length-file
with open(
        "/home/meuleman/projects/ARCH4_data/WM20221126_Tx_length_normalization/transcriptLengthPerGene_WM20221126.txt",
        "r") as f:
    lines = f.readlines()[:]

# Extract the data from the lines
genes_data = [line.strip().split("\t") for line in lines]

# Extract the data from the genes_data list and store them in separate lists
wm_genes_symbol, wm_genes_id, wm_gene_length = zip(*genes_data)

# Create a dictionary mapping gene symbols to gene lengths
wm_dict = {gene_symbol: gene_length for gene_symbol, _, gene_length in genes_data}

# Print the lengths of the lists and the min gene length
print("WM Gene File Length: ", len(genes_data))
print("First Three Gene Data:\n", genes_data[0:3])
print("Min Gene Length: ", min(wm_gene_length))

############################################################################################################
# Open the 57k matched genes between the Gene-Length-File & g:Convert-csv

pickle_in = open("/home/lwu/ARCHS4_v2-1/Data Results/matched_genes_bt_gConvertCsv_GLF.pkl", 'rb')
csv_glf_genes_dict = pickle.load(pickle_in)
pickle_in.close()

print("# Matched Genes:", len(csv_glf_genes_dict))

print()
print("Check First 3 Genes in CSV-GLF Dict:")
for key, value in islice(csv_glf_genes_dict.items(), 3):
    print(key, value)

num_rows, num_cols = expression.shape
new_data = []
for gene_row in range(10000):
    # Gene index
    #print(gene_row)

    # Gene name
    gene_name = list(csv_glf_genes_dict.keys())[gene_row]
    #print("gene_name:", gene_name)

    # Gene data
    gene_data = expression[gene_row]
    #print("row:", gene_data[0:5])

    # GLF Gene name
    glf_gene_name = csv_glf_genes_dict[gene_name]
    #print("glf gene_name:", glf_gene_name)

    # Gene length
    gene_length = wm_dict[glf_gene_name]
    #print("length", gene_length)

    #print(type(gene_data))
    #print(type(gene_length))
    # New row
    new_gene_row = gene_data / int(gene_length)
    #print("new_row:", new_gene_row[0:5])
    test = new_gene_row.tolist()
    new_data.append(test)
    #break

pickle_out = open("corrected_matrix_first10000.pkl", 'wb')
pickle.dump(new_data, pickle_out)
pickle_out.close()

In [ ]:
num_rows, num_cols = expression.shape
new_data = []
for gene_row in range(num_rows):
    
    # Gene index
    #print(gene_row)
    
    # Gene name 
    gene_name = list(csv_glf_genes_dict.keys())[gene_row]
    #prinzt("gene_name:", gene_name)
    
    # Gene data
    gene_data = expression[gene_row]
    #print("row:", gene_data[0:5])
        
    # GLF Gene name 
    glf_gene_name = csv_glf_genes_dict[gene_name]
    #print("glf gene_name:", glf_gene_name)
    
    # Gene length 
    gene_length = wm_dict[glf_gene_name]
    #print("length", gene_length)
    
    #print(type(gene_data))
    #print(type(gene_length))
    # New row 
    new_gene_row = gene_data/int(gene_length)
    #print("new_row:", new_gene_row[0:5])
    test = new_gene_row.tolist()
    new_data.append(test)
    #break
pickle_out = open("corrected_selectedSamples_readCounts_matrix.pkl", 'wb')
pickle.dump(new_data, pickle_out)
pickle_out.close()